# Desafio de Previsão de Preço de Mochilas

Este notebook tem como objetivo resolver o desafio de previsão de preços de mochilas do Kaggle. O objetivo é construir um modelo de aprendizado de máquina capaz de prever o preço de mochilas com base em seus atributos, utilizando os dados fornecidos nos arquivos `train.csv` e `test.csv`. A métrica de avaliação utilizada é a raiz do erro quadrático médio (RMSE), e o resultado final será um arquivo `submission.csv` contendo as previsões para o conjunto de teste.


## Carregamento dos Dados

Nesta etapa, carregaremos os dados de treino e teste usando a biblioteca pandas. Isso nos permitirá explorar e preparar os dados para a construção do modelo de aprendizado de máquina.


In [ ]:
import pandas as pd

# Carregar os dados de treino
try:
    train_df = pd.read_csv('train.csv')
    print("Dados de treino carregados com sucesso.")
except FileNotFoundError:
    print("Erro: Arquivo train.csv não encontrado.")
    train_df = None
except Exception as e:
    print(f"Erro ao carregar train.csv: {e}")
    train_df = None

# Carregar os dados de teste
try:
    test_df = pd.read_csv('test.csv')
    print("Dados de teste carregados com sucesso.")
except FileNotFoundError:
    print("Erro: Arquivo test.csv não encontrado.")
    test_df = None
except Exception as e:
    print(f"Erro ao carregar test.csv: {e}")
    test_df = None

# Exibir as primeiras linhas dos DataFrames para verificação
if train_df is not None:
    print("\nPrimeiras linhas do DataFrame de treino:")
    print(train_df.head())

if test_df is not None:
    print("\nPrimeiras linhas do DataFrame de teste:")
    print(test_df.head())


## Análise Exploratória dos Dados (EDA)

Nesta etapa, vamos explorar os dados para entender melhor suas características. Isso inclui verificar os tipos de dados das colunas, identificar valores ausentes, analisar as distribuições das variáveis e procurar por outliers.

Vamos começar com uma visão geral dos DataFrames de treino e teste.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Informações gerais sobre os DataFrames
if train_df is not None:
    print("\nInformações do DataFrame de treino:")
    print(train_df.info())

if test_df is not None:
    print("\nInformações do DataFrame de teste:")
    print(test_df.info())

# Verificação de valores ausentes
if train_df is not None:
    print("\nValores ausentes no DataFrame de treino:")
    print(train_df.isnull().sum())

if test_df is not None:
    print("\nValores ausentes no DataFrame de teste:")
    print(test_df.isnull().sum())

# Análise descritiva das variáveis numéricas
if train_df is not None:
    print("\nAnálise descritiva das variáveis numéricas no DataFrame de treino:")
    print(train_df.describe())

# Distribuição da variável alvo 'Preço'
if train_df is not None:
    plt.figure(figsize=(8, 6))
    sns.histplot(train_df['Preço'], kde=True)
    plt.title('Distribuição da Variável Preço')
    plt.xlabel('Preço')
    plt.ylabel('Frequência')
    plt.show()

# Análise das variáveis categóricas
categorical_cols = ['Marca', 'Material', 'Tamanho', 'À Prova d\'Água', 'Estilo', 'Cor', 'Compartimento para Laptop']

if train_df is not None:
    for col in categorical_cols:
        if col in train_df.columns:
            plt.figure(figsize=(10, 5))
            sns.countplot(data=train_df, x=col)
            plt.title(f'Contagem de {col} no DataFrame de Treino')
            plt.xticks(rotation=45)
            plt.show()

# Correlação entre as variáveis numéricas
if train_df is not None:
    numerical_cols = train_df.select_dtypes(include=['number']).columns
    corr_matrix = train_df[numerical_cols].corr()
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
    plt.title('Matriz de Correlação das Variáveis Numéricas')
    plt.show()


## Tratamento de Valores Ausentes

Nesta etapa, lidaremos com os valores ausentes identificados na etapa anterior. Para as colunas numéricas, imputaremos os valores faltantes utilizando a média da coluna. Para as colunas categóricas, utilizaremos a moda (valor mais frequente).

Este processo será aplicado tanto ao conjunto de treino quanto ao conjunto de teste para garantir a consistência dos dados.

In [ ]:
import numpy as np

# Imputar valores ausentes
if train_df is not None and test_df is not None:
    # Imputar valores numéricos com a média
    for col in train_df.select_dtypes(include=np.number).columns:
        train_df[col].fillna(train_df[col].mean(), inplace=True)
    for col in test_df.select_dtypes(include=np.number).columns:
        test_df[col].fillna(test_df[col].mean(), inplace=True)

    # Imputar valores categóricos com a moda
    for col in train_df.select_dtypes(include='object').columns:
        train_df[col].fillna(train_df[col].mode()[0], inplace=True)
    for col in test_df.select_dtypes(include='object').columns:
        test_df[col].fillna(test_df[col].mode()[0], inplace=True)

    # Verificar se ainda há valores ausentes
    print("\nValores ausentes após a imputação (DataFrame de treino):")
    print(train_df.isnull().sum())
    print("\nValores ausentes após a imputação (DataFrame de teste):")
    print(test_df.isnull().sum())

else:
    print("DataFrames train_df ou test_df não foram carregados corretamente.")


## Engenharia de Features

Nesta etapa, vamos criar novas features a partir das colunas existentes para potencialmente melhorar o desempenho do modelo. As seguintes features serão criadas:

*   **Comprimento do nome da marca:** Comprimento da string na coluna 'Marca'.
*   **Interação entre capacidade de peso e número de compartimentos:** Multiplicação das colunas 'Capacidade de Peso (kg)' e 'Compartimentos'.
*   **Flags para 'À Prova d'Água' e 'Compartimento para Laptop':** Converter colunas booleanas para representação numérica (0 e 1).

Após a criação das features, a coluna 'id' será removida dos DataFrames.

In [ ]:
# Criar cópias dos DataFrames para evitar modificar os originais
train_df_fe = train_df.copy()
test_df_fe = test_df.copy()

# Engenharia de features no DataFrame de treino
train_df_fe['Comprimento_Marca'] = train_df_fe['Marca'].str.len()
train_df_fe['Interacao_Peso_Compartimentos'] = train_df_fe['Capacidade de Peso (kg)'] * train_df_fe['Compartimentos']
train_df_fe['Flag_A_Prova_DAgua'] = train_df_fe['À Prova d\'Água'].map({'Sim': 1, 'Não': 0})
train_df_fe['Flag_Compartimento_Laptop'] = train_df_fe['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0})

# Engenharia de features no DataFrame de teste
test_df_fe['Comprimento_Marca'] = test_df_fe['Marca'].str.len()
test_df_fe['Interacao_Peso_Compartimentos'] = test_df_fe['Capacidade de Peso (kg)'] * test_df_fe['Compartimentos']
test_df_fe['Flag_A_Prova_DAgua'] = test_df_fe['À Prova d\'Água'].map({'Sim': 1, 'Não': 0})
test_df_fe['Flag_Compartimento_Laptop'] = test_df_fe['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0})

# Remover a coluna 'id' dos DataFrames
train_df_fe.drop('id', axis=1, inplace=True)
test_df_fe.drop('id', axis=1, inplace=True)

# Exibir as primeiras linhas dos DataFrames com as novas features
print("\nPrimeiras linhas do DataFrame de treino com novas features:")
print(train_df_fe.head())

print("\nPrimeiras linhas do DataFrame de teste com novas features:")
print(test_df_fe.head())

# Atualizar os DataFrames originais
train_df = train_df_fe
test_df = test_df_fe


## Codificação de Variáveis Categóricas

Nesta etapa, vamos codificar as variáveis categóricas usando a técnica de one-hot encoding. Isso transformará cada valor único em uma coluna separada, com valores 0 ou 1 indicando a presença ou ausência daquela categoria. Essa etapa é crucial para que os modelos de machine learning possam processar as variáveis categóricas corretamente.

As colunas a serem codificadas são: 'Marca', 'Material', 'Tamanho', 'Estilo' e 'Cor'. As colunas 'À Prova d\'Água' e 'Compartimento para Laptop' já foram convertidas para representação numérica na etapa anterior.

É importante aplicar a mesma transformação tanto no conjunto de treino quanto no conjunto de teste para garantir a consistência dos dados.

In [ ]:
import pandas as pd

# Identificar as colunas categóricas
categorical_cols = ['Marca', 'Material', 'Tamanho', 'Estilo', 'Cor']

# Criar cópias dos DataFrames para evitar modificar os originais
train_df_encoded = train_df.copy()
test_df_encoded = test_df.copy()

# Aplicar one-hot encoding nas colunas categóricas do DataFrame de treino
train_df_encoded = pd.get_dummies(train_df_encoded, columns=categorical_cols, dummy_na=False)

# Aplicar one-hot encoding nas colunas categóricas do DataFrame de teste
test_df_encoded = pd.get_dummies(test_df_encoded, columns=categorical_cols, dummy_na=False)

# Exibir as primeiras linhas dos DataFrames transformados
print("\nPrimeiras linhas do DataFrame de treino após one-hot encoding:")
print(train_df_encoded.head())

print("\nPrimeiras linhas do DataFrame de teste após one-hot encoding:")
print(test_df_encoded.head())

# Atualizar os DataFrames originais
train_df = train_df_encoded
test_df = test_df_encoded


## Divisão dos Dados em Treino e Validação

Nesta etapa, dividiremos o conjunto de dados de treino em dois subconjuntos: um conjunto de treino e um conjunto de validação. O conjunto de treino será usado para treinar o modelo de aprendizado de máquina, enquanto o conjunto de validação será usado para avaliar o desempenho do modelo e ajustar seus hiperparâmetros. Uma divisão comum é usar 80% dos dados para treino e 20% para validação.

Utilizaremos a função `train_test_split` da biblioteca scikit-learn para realizar essa divisão. É importante garantir que a divisão seja feita de forma aleatória para evitar qualquer viés nos resultados.

In [ ]:
from sklearn.model_selection import train_test_split

# Separar a variável alvo (Preço) das features
X = train_df.drop('Preço', axis=1)
y = train_df['Preço']

# Dividir os dados em treino e validação (80% treino, 20% validação)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Exibir o tamanho dos conjuntos de treino e validação
print("Tamanho do conjunto de treino:", X_train.shape)
print("Tamanho do conjunto de validação:", X_val.shape)


## Treinamento do Modelo XGBoost

Nesta etapa, treinaremos um modelo XGBoost para prever o preço das mochilas. Usaremos os dados de treino (`X_train`, `y_train`) para treinar o modelo e o conjunto de validação (`X_val`, `y_val`) para monitorar o desempenho e evitar overfitting. Definiremos alguns hiperparâmetros iniciais para o modelo e o treinaremos por um número fixo de rodadas.

Após o treinamento, avaliaremos o modelo no conjunto de validação para ter uma ideia do seu desempenho antes de fazer previsões no conjunto de teste.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Definir os hiperparâmetros do XGBoost
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Converter os dados em formato DMatrix (formato otimizado do XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Definir a lista de avaliação para monitorar o desempenho no conjunto de validação
evals = [(dtrain, 'train'), (dval, 'val')]

# Treinar o modelo XGBoost
model = xgb.train(params,
                  dtrain,
                  num_boost_round=1000,
                  evals=evals,
                  early_stopping_rounds=50,
                  verbose_eval=100)

# Fazer previsões no conjunto de validação
y_pred = model.predict(dval)

# Calcular o RMSE no conjunto de validação
rmse = mean_squared_error(y_val, y_pred, squared=False)
print("\nRMSE no conjunto de validação:", rmse)


In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Converter colunas 'Compartimento para Laptop' e 'À Prova d'Água' para tipo numérico (int)
X_train['Compartimento para Laptop'] = X_train['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
X_train['À Prova d\'Água'] = X_train['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)
X_val['Compartimento para Laptop'] = X_val['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
X_val['À Prova d\'Água'] = X_val['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)

# Definir os hiperparâmetros do XGBoost
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Converter os dados em formato DMatrix (formato otimizado do XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Definir a lista de avaliação para monitorar o desempenho no conjunto de validação
evals = [(dtrain, 'train'), (dval, 'val')]

# Treinar o modelo XGBoost
model = xgb.train(params,
                  dtrain,
                  num_boost_round=1000,
                  evals=evals,
                  early_stopping_rounds=50,
                  verbose_eval=100)

# Fazer previsões no conjunto de validação
y_pred = model.predict(dval)

# Calcular o RMSE no conjunto de validação
rmse = mean_squared_error(y_val, y_pred, squared=False)
print("\nRMSE no conjunto de validação:", rmse)


In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Converter colunas 'Compartimento para Laptop' e 'À Prova d'Água' para tipo numérico (int)
X_train['Compartimento para Laptop'] = X_train['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
X_train['À Prova d\'Água'] = X_train['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)
X_val['Compartimento para Laptop'] = X_val['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
X_val['À Prova d\'Água'] = X_val['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)

# Definir os hiperparâmetros do XGBoost
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Converter os dados em formato DMatrix (formato otimizado do XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Definir a lista de avaliação para monitorar o desempenho no conjunto de validação
evals = [(dtrain, 'train'), (dval, 'val')]

# Treinar o modelo XGBoost
model = xgb.train(params,
                  dtrain,
                  num_boost_round=1000,
                  evals=evals,
                  early_stopping_rounds=50,
                  verbose_eval=100)

# Fazer previsões no conjunto de validação
y_pred = model.predict(dval)

# Calcular o RMSE no conjunto de validação
rmse = mean_squared_error(y_val, y_pred)
rmse = rmse**0.5
print("\nRMSE no conjunto de validação:", rmse)


In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Converter colunas 'Compartimento para Laptop' e 'À Prova d'Água' para tipo numérico (int)
# X_train['Compartimento para Laptop'] = X_train['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
# X_train['À Prova d\'Água'] = X_train['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)
# X_val['Compartimento para Laptop'] = X_val['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
# X_val['À Prova d\'Água'] = X_val['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)

# Definir os hiperparâmetros do XGBoost
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Converter os dados em formato DMatrix (formato otimizado do XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Definir a lista de avaliação para monitorar o desempenho no conjunto de validação
evals = [(dtrain, 'train'), (dval, 'val')]

# Treinar o modelo XGBoost
model = xgb.train(params,
                  dtrain,
                  num_boost_round=1000,
                  evals=evals,
                  early_stopping_rounds=50,
                  verbose_eval=100)

# Fazer previsões no conjunto de validação
y_pred = model.predict(dval)

# Calcular o RMSE no conjunto de validação
rmse = mean_squared_error(y_val, y_pred)
rmse = rmse**0.5
print("\nRMSE no conjunto de validação:", rmse)


## Avaliação do Modelo no Conjunto de Validação

Agora que o modelo foi treinado, vamos avaliar seu desempenho no conjunto de validação. Já calculamos as previsões (`y_pred`) e o RMSE na célula anterior. Aqui, vamos apenas apresentar o resultado de forma mais clara.

O RMSE (Root Mean Squared Error) é uma métrica comum para avaliar modelos de regressão. Quanto menor o RMSE, melhor o desempenho do modelo. Neste caso, o RMSE obtido no conjunto de validação é de 38.93.

Este valor servirá como base para compararmos com o desempenho de modelos ajustados posteriormente.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Converter colunas 'Compartimento para Laptop' e 'À Prova d'Água' para tipo numérico (int)
# X_train['Compartimento para Laptop'] = X_train['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
# X_train['À Prova d\'Água'] = X_train['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)
# X_val['Compartimento para Laptop'] = X_val['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
# X_val['À Prova d\'Água'] = X_val['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)

# Definir os hiperparâmetros do XGBoost
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Converter os dados em formato DMatrix (formato otimizado do XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Definir a lista de avaliação para monitorar o desempenho no conjunto de validação
evals = [(dtrain, 'train'), (dval, 'val')]

# Treinar o modelo XGBoost
# model = xgb.train(params,
#                   dtrain,
#                   num_boost_round=1000,
#                   evals=evals,
#                   early_stopping_rounds=50,
#                   verbose_eval=100)

# Fazer previsões no conjunto de validação
y_pred = model.predict(dval)

# Calcular o RMSE no conjunto de validação
rmse = mean_squared_error(y_val, y_pred)
rmse = rmse**0.5
print("\nRMSE no conjunto de validação:", rmse)


## Ajuste de Hiperparâmetros

Nesta etapa, vamos otimizar os hiperparâmetros do modelo XGBoost utilizando `GridSearchCV`. Ajustaremos `max_depth` e `eta` para melhorar o desempenho do modelo. O `GridSearchCV` irá testar diferentes combinações desses hiperparâmetros e selecionar a que apresentar o melhor resultado no conjunto de validação.

**Importante:** O treinamento do modelo foi comentado na célula anterior. Vamos descomentá-lo e executar o `GridSearchCV` para encontrar os melhores hiperparâmetros.

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Definir os hiperparâmetros a serem ajustados
param_grid = {
    'max_depth': [3, 5, 7],
    'eta': [0.01, 0.1, 0.3]
}

# Definir o modelo XGBoost para o GridSearchCV
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', 
                             eval_metric='rmse',
                             subsample=0.8,
                             colsample_bytree=0.8,
                             seed=42)

# Converter os dados em formato DMatrix (formato otimizado do XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Configurar o GridSearchCV
grid_search = GridSearchCV(estimator=xgb_model,
                           param_grid=param_grid,
                           scoring='neg_root_mean_squared_error',
                           cv=3,  # Usar validação cruzada com 3 folds
                           verbose=2,
                           n_jobs=-1)  # Usar todos os núcleos da CPU

# Executar o GridSearchCV
grid_search.fit(X_train, y_train)

# Exibir os melhores hiperparâmetros encontrados
print("Melhores hiperparâmetros:", grid_search.best_params_)

# Exibir o melhor score (RMSE negativo) encontrado
print("Melhor score (RMSE negativo):", grid_search.best_score_)

# Obter o melhor modelo treinado
best_model = grid_search.best_estimator_

# Avaliar o melhor modelo no conjunto de validação
y_pred = best_model.predict(X_val)
rmse = mean_squared_error(y_val, y_pred)**0.5
print("RMSE no conjunto de validação com o melhor modelo:", rmse)


## Treinamento do Modelo Final

Agora que encontramos os melhores hiperparâmetros, vamos treinar o modelo final usando todo o conjunto de dados de treino. Isso garantirá que o modelo tenha o máximo de informações possível para fazer previsões precisas no conjunto de teste.

Usaremos os hiperparâmetros otimizados (`eta`: 0.1, `max_depth`: 3) e treinaremos o modelo XGBoost com todos os dados de treino. Em seguida, salvaremos o modelo treinado para uso posterior na etapa de previsão.

In [ ]:
import xgboost as xgb

# Definir os melhores hiperparâmetros encontrados
best_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Criar DMatrix com todos os dados de treino
X_train_all = train_df.drop('Preço', axis=1)
y_train_all = train_df['Preço']
dtrain_all = xgb.DMatrix(X_train_all, label=y_train_all)

# Treinar o modelo final com todos os dados de treino
final_model = xgb.train(
    best_params,
    dtrain_all,
    num_boost_round=1000,  # Manter um número alto de rodadas
    verbose_eval=False # Desativar o output detalhado
)

print("Modelo final treinado com sucesso.")


In [ ]:
import xgboost as xgb

# Definir os melhores hiperparâmetros encontrados
best_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Criar DMatrix com todos os dados de treino
X_train_all = train_df.drop('Preço', axis=1)
y_train_all = train_df['Preço']

# Converter 'Compartimento para Laptop' e 'À Prova d\'Água' para tipo numérico (int)
X_train_all['Compartimento para Laptop'] = X_train_all['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
X_train_all['À Prova d\'Água'] = X_train_all['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)

dtrain_all = xgb.DMatrix(X_train_all, label=y_train_all)

# Treinar o modelo final com todos os dados de treino
final_model = xgb.train(
    best_params,
    dtrain_all,
    num_boost_round=1000,  # Manter um número alto de rodadas
    verbose_eval=False # Desativar o output detalhado
)

print("Modelo final treinado com sucesso.")


## Previsões no Conjunto de Teste

Nesta etapa, utilizaremos o modelo final treinado para fazer previsões no conjunto de teste. É crucial garantir que o conjunto de teste seja pré-processado da mesma forma que o conjunto de treino, incluindo a codificação de variáveis categóricas e o tratamento de valores ausentes. Após o pré-processamento, converteremos o conjunto de teste para o formato `DMatrix` do XGBoost e utilizaremos o modelo treinado para gerar as previsões.

As previsões geradas serão então utilizadas para criar o arquivo de submissão no formato exigido pela competição.

In [ ]:
import xgboost as xgb
import pandas as pd

# Converter o conjunto de teste para o formato DMatrix
X_test = test_df.copy()

# Converter 'Compartimento para Laptop' e 'À Prova d\'Água' para tipo numérico (int)
X_test['Compartimento para Laptop'] = X_test['Compartimento para Laptop'].map({'Sim': 1, 'Não': 0}).astype(int)
X_test['À Prova d\'Água'] = X_test['À Prova d\'Água'].map({'Sim': 1, 'Não': 0}).astype(int)

dtest = xgb.DMatrix(X_test)

# Fazer as previsões no conjunto de teste
test_predictions = final_model.predict(dtest)

# Exibir as primeiras previsões
print("Primeiras previsões no conjunto de teste:\n", test_predictions[:10])


## Criação do Arquivo de Submissão

Nesta etapa final, criaremos o arquivo de submissão no formato exigido pela competição. O arquivo deve conter duas colunas: `id` e `Preço`, onde `id` corresponde aos IDs do conjunto de teste e `Preço` corresponde às previsões geradas pelo modelo.

Primeiro, criaremos um DataFrame pandas com as colunas `id` e `Preço`. Em seguida, salvaremos este DataFrame em um arquivo CSV chamado `submission.csv`, garantindo que o arquivo inclua o cabeçalho e esteja no formato correto para a submissão.

In [ ]:
import pandas as pd

# Carregar o DataFrame de teste original para obter os IDs
test_df_original = pd.read_csv('test.csv')

# Criar um DataFrame com os IDs e as previsões
submission_df = pd.DataFrame({'id': test_df_original['id'], 'Preço': test_predictions})

# Converter a coluna 'id' para inteiro
submission_df['id'] = submission_df['id'].astype(int)

# Formatar a coluna 'Preço' para 5 casas decimais
submission_df['Preço'] = submission_df['Preço'].round(5)

# Salvar o DataFrame em um arquivo CSV
submission_df.to_csv('submission.csv', index=False)

# Exibir as primeiras linhas do arquivo de submissão
print("Primeiras linhas do arquivo de submissão:\n", submission_df.head())

print("Arquivo de submissão 'submission.csv' criado com sucesso.")
